# Sequential & session-based recommendation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Using order and recency instead of only long-term taste
Sequential recommenders predict the next item from a recent trajectory. The examples compute recency weights, a Markov transition table, next-item probabilities, a session embedding, and an attention-weighted session summary.

In [ ]:
# 1) recency weighting gives the latest actions more influence
items=np.array([[1,0],[0,1],[1,1]],float); weights=np.array([0.2,0.3,0.5]); profile=weights@items
plt.figure(figsize=(6,3)); plt.bar(['dim0','dim1'],profile); plt.title('recency-weighted session profile')
assert np.allclose(profile,[0.7,0.8])

In [ ]:
# 2) first-order Markov counts estimate next-item transitions
sessions=[[0,1,2],[0,1,1],[1,2,0]]; C=np.zeros((3,3),int)
for s in sessions:
    for a,b in zip(s,s[1:]): C[a,b]+=1
P=C/C.sum(axis=1,keepdims=True)
plt.figure(figsize=(4,4)); plt.imshow(P,cmap='Blues',vmin=0,vmax=1); plt.colorbar(label='P(next|current)'); plt.title('transition probabilities')
assert abs(P[1,2]-2/3)<1e-12

In [ ]:
# 3) next-item prediction from the current last item
P=np.array([[0,1,0],[0,1/3,2/3],[1,0,0]],float); last=1; probs=P[last]
plt.figure(figsize=(6,3)); plt.bar(['item0','item1','item2'],probs); plt.title('next after item1')
assert int(np.argmax(probs))==2 and abs(probs[2]-2/3)<1e-12

In [ ]:
# 4) session embedding by averaging recent item embeddings
E=np.array([[1,0],[0,1],[1,1]],float); sess=[0,1,2]; emb=E[sess].mean(axis=0)
plt.figure(figsize=(5,4)); plt.scatter(E[:,0],E[:,1]); plt.scatter([emb[0]],[emb[1]],c='r',s=80); plt.title(f'session embedding {emb}')
assert np.allclose(emb,[2/3,2/3])

In [ ]:
# 5) attention makes the query choose which past event matters
E=np.array([[1,0],[0,1],[1,1]],float); query=np.array([1,0.2]); dots=E@query; a=np.exp(dots)/np.exp(dots).sum(); ctx=a@E
plt.figure(figsize=(6,3)); plt.bar(['event0','event1','event2'],a); plt.title(f'attention context=({ctx[0]:.3f},{ctx[1]:.3f})')
assert a[2]>a[1] and abs(ctx[0]-0.831758105417318)<1e-9